# text-analysis-demo

A reproducible text analysis pipeline for humanities researchers.

---
*Auto-generated from `codemeta.json`*

## 1. Environment check

In [ ]:
import sys
print(f"Python {sys.version}")
print("Environment ready.")

## 2. Import dependencies

In [ ]:
import nltk
import pandas as pd
import matplotlib.pyplot as plt
from wordcloud import WordCloud
print('All imports successful.')

## 3. Text cleaning utility

Run this cell first before pasting any text.

In [ ]:
import unicodedata

def clean_text(text):
    text = unicodedata.normalize('NFKC', text)
    replacements = {
        chr(0x2018): chr(39), chr(0x2019): chr(39),
        chr(0x201c): chr(34), chr(0x201d): chr(34),
        chr(0x2013): '-', chr(0x2014): '-',
        chr(0x2026): '...', chr(0x00a0): ' ',
        chr(0x00ad): '',
    }
    for k, v in replacements.items():
        text = text.replace(k, v)
    return text

print('clean_text() ready.')

## 4. Dataset

Upload a .txt or .pdf file, or paste your text below.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import io

upload = widgets.FileUpload(accept='.txt,.pdf', multiple=False)
display(upload)
print('Upload a .txt or .pdf file above, then run the next cell.')

In [ ]:
if upload.value:
    val = list(upload.value.values())[0] if isinstance(upload.value, dict) else upload.value[0]
    fname = val['metadata']['name'] if isinstance(upload.value, dict) else val['name']
    raw = bytes(val['content'])
    if fname.endswith('.pdf'):
        import subprocess, sys
        subprocess.run([sys.executable, '-m', 'pip', 'install', 'pypdf2', '-q'])
        import PyPDF2, io as _io
        reader = PyPDF2.PdfReader(_io.BytesIO(raw))
        text = ' '.join(page.extract_text() or '' for page in reader.pages)
    else:
        text = raw.decode('utf-8', errors='ignore')
    corpus = clean_text(text)
    print(f'Loaded: {fname} — {len(corpus.split())} words')
else:
    corpus = clean_text("""
    Paste your text here. Any paragraph from an article, paper,
    or any source relevant to your research. At least 50 words
    works best for meaningful results.
    """)
    print(f'Corpus loaded: {len(corpus.split())} words')

## 5. Text analysis

Run all cells to generate the word frequency chart and word cloud.

In [ ]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from collections import Counter
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

stop_words = set(stopwords.words('english'))
tokens = word_tokenize(corpus.lower())
words = [w for w in tokens if w.isalpha() and w not in stop_words]
freq = Counter(words)

top_words = freq.most_common(15)
df = pd.DataFrame(top_words, columns=['word', 'count'])
fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(df['word'][::-1], df['count'][::-1], color='#534AB7', alpha=0.8)
ax.set_xlabel('Frequency')
ax.set_title('Top 15 words in corpus')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

text_clean = ' '.join(words)
wc = WordCloud(width=800, height=400, background_color='white', colormap='viridis', max_words=80).generate(text_clean)
plt.figure(figsize=(12, 5))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title('Word cloud')
plt.tight_layout()
plt.savefig('wordcloud.png')
plt.show()

freq_df = pd.DataFrame(freq.most_common(5), columns=['word', 'count'])
print('Top 5 words:')
print(freq_df.to_string())
print('Word cloud saved as wordcloud.png')